# Water, Energy, and the Material Conditions of Life Workflow

This notebook scaffold supports the article **Water, Energy, and the Material Conditions of Life**. It can be expanded with osmotic pressure, water potential, homeostatic setpoint dynamics, growth fitting, oxygen limitation, energy allocation, material-condition scoring, and provenance notes.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

article_dir = Path.cwd().parent
solute = pd.read_csv(article_dir / 'data' / 'solute_conditions.csv')
R_GAS = 0.082057
solute['osmotic_pressure_atm'] = solute['van_t_hoff_factor'] * solute['concentration_mol_L'] * R_GAS * solute['temperature_K']
solute['relative_water_stress'] = solute['osmotic_pressure_atm'] / solute['osmotic_pressure_atm'].max()
solute.round(5)

In [ ]:
wp = pd.read_csv(article_dir / 'data' / 'water_potential_scenarios.csv')
components = ['solute_potential_MPa','pressure_potential_MPa','gravitational_potential_MPa','matric_potential_MPa']
wp['total_water_potential_MPa'] = wp[components].sum(axis=1)
wp.round(5)

In [ ]:
growth = pd.read_csv(article_dir / 'data' / 'growth_observations.csv')
rows = []
for condition, group in growth.groupby('condition'):
    slope, intercept = np.polyfit(group['time_h'], np.log(group['abundance']), 1)
    rows.append({'condition': condition, 'growth_rate_per_h': slope, 'N0': np.exp(intercept), 'doubling_time_h': np.log(2) / slope})
pd.DataFrame(rows).round(5)

In [ ]:
oxygen = pd.read_csv(article_dir / 'data' / 'oxygen_scenarios.csv')
oxygen['relative_energy_rate'] = oxygen['max_relative_energy_rate'] * oxygen['oxygen_mg_L'] / (oxygen['half_saturation_mg_L'] + oxygen['oxygen_mg_L'])
oxygen['oxygen_limitation'] = 1 - oxygen['relative_energy_rate']
oxygen.round(5)

In [ ]:
condition = pd.read_csv(article_dir / 'data' / 'material_condition_sites.csv')
condition['material_condition_score'] = (
    0.17 * condition['water_availability'] +
    0.15 * condition['osmotic_stability'] +
    0.17 * condition['energy_availability'] +
    0.14 * condition['oxygen_support'] +
    0.13 * condition['thermal_suitability'] +
    0.14 * condition['ph_stability'] +
    0.10 * (1 - condition['stress_penalty'])
)
condition.sort_values('material_condition_score', ascending=False).round(3)